In [ ]:
### imports
import os
import sqlite3
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
### loding env variables and initializations
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"Openai key exits and starts with sk-proj-xxxx")
else:
    print(f"Openai key does not exist")

client = OpenAI()

In [ ]:
### setting up the database
def init_db():
    conn = sqlite3.connect("airline.db")#connecting to airline db file
    cursor=conn.cursor() # creating cursor obj to execute sql commands
    cursor.execute("CREATE TABLE IF NOT EXISTS FLIGHTS (id TEXT, destination TEXT, status TEXT)") # creating flight table
    cursor.execute("INSERT OR IGNORE INTO flights VALUES ('AI101', 'London', 'On Time'), ('AI202', 'New York', 'Delayed')") # Add sample flight data
    conn.commit() ## save the changes to the db
    conn.close()

init_db() ## run the database initialization

In [ ]:
### defining the tools
## tool no 1: definig a tool to check flight status
def getFlightStatus(flightId):
    conn=sqlite3.connect("airline.db")
    cursor=conn.cursor()
    cursor.execute("SELECT status FROM FLIGHTS WHERE id=?",(flightID,))
    result=cursor.fetchone() # fetch the first matching row
    conn.close() # close the connection
    return result[0] if result else "Flight not found" # return status

## tool no 2: defining a tool funtion to simulate booking
def bookFlight(flightId, destination):
    conn=sqlite3.connect("airline.db")
    cursor=conn.cursor()
    cursor.execute("INSERT INTO flights VALUES (?,?,?)", (flightId, destination, "Confirmed"))
    conn.commit() # commit new row to db
    conn.close() ## close the connection
    return f"successfully booked {flightId} to {destination}"


In [ ]:
# writing metadata for openai to understand available tools
TOOLS = [
    {
        # describe about first tool
        "type": "function",
        "function": {  # defining function details
            "name": "getFlightStatus",
            "description": "Get the current status of a flight",
            "parameters": {  # defining expected arguments
                "type": "object",  # args are passed as objects
                "properties": {
                    "flightId": {
                        "type": "string",
                        "description": "The flight ID, e.g. AI101"
                    }
                },
                "required": ["flightId"]  # marking it as mandatory
            }
        }
    },
    ## describing about second tool
    { 
        "type":"function",
        "function":{
            "name":"bookFlight",
            "description":"Book a new flight for the user",
            "parameters":{
                "type":"object",
                "properties":{
                    "flightId":{"type":"string","description":"The flight ID"},
                    "destination":{"type":"string","description":"The destination city"}
                },
                "required":["flightId","destination"]
            }
        }
    }
]

In [ ]:
## creating core logic
# defining main logic function for gradio
def airlineBot(userInput, chatHistory):
    messages = [{"role":"system","content":"You are a very helpful AI Assistant. Use the tools to check the status or to book flights"}]
    
    ## looping through previous chat history
    for userMsg, botMsg in chatHistory:
        messages.append({"role":"user","content":userMsg}) ## append past user messages
        messages.append({"role":"system","content":botMsg}) ## appending past bot messages
    
    messages.append({"role":"user", "content":userInput}) ## append the current user input

    ## finding the response
    response = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=messages,
        tools=TOOLS
    )
    
    responseMsg = response.choices[0].message # extracting the reply from the chat model
    toolCalls = responseMsg.tool_calls # capturing the requested tool calls

    ## checking if the model wants to call any tools
    if toolCalls: # Check if the model wants to call any tools
        messages.append(responseMsg) # Add the model's tool request to the message history
        
        for tool_call in toolCalls: # Iterate through each requested tool call (Parallel Handling)
            function_name = tool_call.function.name # Get the name of the function to call
            
            try:
                args = json.loads(tool_call.function.arguments) # Parse the arguments from JSON
            except:
                args = {}
            
            if function_name == "getFlightStatus": # Check if the model called getFlightStatus
                result = getFlightStatus(args.get("flightId")) # Execute the status check function
            elif function_name == "bookFlight": # Check if the model called bookFlight
                result = bookFlight(args.get("flightId"), args.get("destination")) # Execute the booking function
            else:
                result = "Unknown function"
            
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": str(result)
            }) # Provide tool result back to the model

        second_response = client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=messages
        ) # Get a final response from the model
        
        return second_response.choices[0].message.content # Return the final natural language answer
    
    return responseMsg.content # Return the model's direct response if no tools were used

In [ ]:
### settigng up the UI
with gr.Blocks() as demo: # Initialize the Gradio Blocks layout
    gr.Markdown("#  Frontier Airline Assistant") # Display a markdown heading
    chatbot = gr.Chatbot() # Create a chatbot UI component
    msg = gr.Textbox(label="Ask about flight status or book a flight") # Create a textbox for user input
    clear = gr.Button("Clear") # Create a button to clear the chat

    def respond(message, chat_history): # Define the UI response wrapper
        bot_message = airlineBot(message, chat_history) # Get the bot's response from the logic function
        chat_history.append((message, bot_message)) # Update the history with the new exchange
        return "", chat_history # Reset the input box and return the updated history

    msg.submit(respond, [msg, chatbot], [msg, chatbot]) # Trigger response when user presses Enter
    clear.click(lambda: None, None, chatbot, queue=False) # Clear the chatbot state when button is clicked

demo.launch() # Launch the web server and open the UI